> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# Agent as Tool Pattern - Complete Multi-Agent System


In [ ]:
!pip install --upgrade boto3 botocore strands-agents -q
print('After installation, restart the kernel (Kernel > Restart Kernel) and run from the next cell')

## Create AWS Resources (S3 Vectors, DynamoDB)

In [ ]:
import boto3

REGION = 'us-east-1'
ACCOUNT_ID = boto3.client('sts', region_name=REGION).get_caller_identity()['Account']

# --- Video S3 URI ---
S3_BUCKET_NAME = 'twelvelabs-video-demo-bucket'  # TODO: Replace with your S3 bucket containing videos
VIDEO_S3_URI = f's3://{S3_BUCKET_NAME}/videos/redbull1transcode.mp4'  # TODO: Replace with your video path

s3vectors_setup = boto3.client('s3vectors', region_name=REGION)
dynamodb_setup = boto3.client('dynamodb', region_name=REGION)

# --- S3 Vectors Bucket ---
S3_VECTORS_BUCKET = f'video-vectors-{ACCOUNT_ID}'
try:
    s3vectors_setup.create_vector_bucket(vectorBucketName=S3_VECTORS_BUCKET)
    print(f'Created S3 Vectors bucket: {S3_VECTORS_BUCKET}')
except s3vectors_setup.exceptions.ConflictException:
    print(f'S3 Vectors bucket already exists: {S3_VECTORS_BUCKET}')

# --- S3 Vectors Index ---
S3_VECTORS_INDEX = 'marengo-index'
try:
    s3vectors_setup.create_index(
        vectorBucketName=S3_VECTORS_BUCKET,
        indexName=S3_VECTORS_INDEX,
        dataType='float32',
        dimension=512,
        distanceMetric='cosine'
    )
    print(f'Created S3 Vectors index: {S3_VECTORS_INDEX}')
except s3vectors_setup.exceptions.ConflictException:
    print(f'S3 Vectors index already exists: {S3_VECTORS_INDEX}')

# --- DynamoDB Table ---
DYNAMODB_TABLE = 'video-tasks'
try:
    dynamodb_setup.create_table(
        TableName=DYNAMODB_TABLE,
        KeySchema=[{'AttributeName': 'task_id', 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': 'task_id', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST'
    )
    waiter = dynamodb_setup.get_waiter('table_exists')
    waiter.wait(TableName=DYNAMODB_TABLE)
    print(f'Created DynamoDB table: {DYNAMODB_TABLE}')
except dynamodb_setup.exceptions.ResourceInUseException:
    print(f'DynamoDB table already exists: {DYNAMODB_TABLE}')

print(f'\n--- Resource Summary ---')
print(f'VIDEO_S3_URI      = {VIDEO_S3_URI}')
print(f'S3_VECTORS_BUCKET = {S3_VECTORS_BUCKET}')
print(f'S3_VECTORS_INDEX  = {S3_VECTORS_INDEX}')
print(f'DYNAMODB_TABLE    = {DYNAMODB_TABLE}')

In [ ]:
import json, time, uuid, hashlib
from strands import Agent, tool
from strands.models import BedrockModel
from strands.handlers.callback_handler import PrintingCallbackHandler
from strands.agent.conversation_manager import SlidingWindowConversationManager
from typing import Dict, Any
from botocore.config import Config

# S3_VECTORS_BUCKET, S3_VECTORS_INDEX, DYNAMODB_TABLE, REGION, ACCOUNT_ID
# are set by the resource creation cell above

MARENGO_MODEL_ID = 'twelvelabs.marengo-embed-3-0-v1:0'
PEGASUS_MODEL_ID = 'us.twelvelabs.pegasus-1-2-v1:0'

bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client('s3', region_name=REGION)
s3vectors = boto3.client('s3vectors', region_name=REGION)
dynamodb = boto3.resource('dynamodb', region_name=REGION)
task_table = dynamodb.Table(DYNAMODB_TABLE)
transcribe = boto3.client('transcribe', region_name=REGION)
s3_sigv4 = boto3.client('s3', region_name=REGION, config=Config(signature_version='s3v4'))

claude_model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0', region_name=REGION)

print(f'S3_VECTORS_BUCKET = {S3_VECTORS_BUCKET}')
print(f'S3_VECTORS_INDEX  = {S3_VECTORS_INDEX}')
print(f'DYNAMODB_TABLE    = {DYNAMODB_TABLE}')
print('Initialization complete')

## Step 1: Define All Base Tools

In [ ]:
# Embedding tool
@tool
def create_video_embedding(s3_uri: str) -> Dict[str, Any]:
    '''Generate video embeddings (waits for completion, then stores in S3 Vectors).'''
    task_id = str(uuid.uuid4())[:8]
    bucket = s3_uri.split('/')[2]
    key = '/'.join(s3_uri.split('/')[3:])
    task_table.put_item(Item={'task_id': task_id, 's3_uri': s3_uri, 's3_bucket': bucket, 's3_key': key, 'status': 'processing', 'created_at': int(time.time())})
    
    response = bedrock_runtime.start_async_invoke(
        modelId=MARENGO_MODEL_ID,
        modelInput={'inputType': 'video', 'video': {'mediaSource': {'s3Location': {'uri': s3_uri, 'bucketOwner': ACCOUNT_ID}}, 'embeddingOption': ['visual', 'audio'], 'embeddingScope': ['clip'], 'segmentation': {'method': 'fixed', 'fixed': {'durationSec': 6}}}},
        outputDataConfig={'s3OutputDataConfig': {'s3Uri': f's3://{bucket}/embeddings/{task_id}/'}}
    )
    invocation_arn = response['invocationArn']
    task_table.update_item(Key={'task_id': task_id}, UpdateExpression='SET invocation_arn = :arn', ExpressionAttributeValues={':arn': invocation_arn})
    
    print(f'Processing embeddings... (task_id: {task_id})')
    for _ in range(60):
        status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)['status']
        if status == 'Completed': break
        if status in ['Failed', 'Expired']: return {'error': f'Embedding failed: {status}', 'task_id': task_id}
        time.sleep(10)
    else:
        return {'error': 'Timeout', 'task_id': task_id}
    
    output_uri = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)['outputDataConfig']['s3OutputDataConfig']['s3Uri']
    prefix = '/'.join(output_uri.split('/')[3:])
    objs = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    json_key = next((o['Key'] for o in objs.get('Contents', []) if o['Key'].endswith('output.json')), None)
    data = json.loads(s3.get_object(Bucket=bucket, Key=json_key)['Body'].read())
    clips = data.get('data', [])
    
    vectors = [{'key': f"{task_id}_{c['embeddingOption']}_{c['startSec']}_{c['endSec']}", 'data': {'float32': c['embedding']}, 'metadata': {'task_id': task_id, 'embeddingOption': c['embeddingOption'], 'startSec': c['startSec'], 'endSec': c['endSec']}} for c in clips]
    for i in range(0, len(vectors), 100): s3vectors.put_vectors(vectorBucketName=S3_VECTORS_BUCKET, indexName=S3_VECTORS_INDEX, vectors=vectors[i:i+100])
    
    task_table.update_item(Key={'task_id': task_id}, UpdateExpression='SET #s = :s, clip_count = :c', ExpressionAttributeNames={'#s': 'status'}, ExpressionAttributeValues={':s': 'completed', ':c': len(clips)})
    return {'task_id': task_id, 'status': 'completed', 's3_uri': s3_uri, 'stored_clips': len(clips)}

print('Embedding tool defined')

In [ ]:
# Search tools (up to 10 results)
@tool
def search_video_clips(query: str, top_k: int = 50, max_results: int = 10) -> Dict[str, Any]:
    '''Search video clips by text. Returns up to 10 deduplicated results.'''
    response = bedrock_runtime.invoke_model(modelId=MARENGO_MODEL_ID, body=json.dumps({'inputType': 'text', 'text': {'inputText': query}}), contentType='application/json')
    emb = json.loads(response['body'].read())['data'][0]['embedding']
    
    results = s3vectors.query_vectors(vectorBucketName=S3_VECTORS_BUCKET, indexName=S3_VECTORS_INDEX, queryVector={'float32': emb}, topK=top_k, returnDistance=True, returnMetadata=True, filter={'embeddingOption': {'$in': ['visual', 'audio']}})
    
    clips = []
    seen_intervals = []
    
    for v in results.get('vectors', []):
        if len(clips) >= max_results:
            break
            
        meta = v.get('metadata', {})
        task_info = task_table.get_item(Key={'task_id': meta.get('task_id')}).get('Item', {})
        start, end = int(meta.get('startSec', 0)), int(meta.get('endSec', 0))
        
        # Deduplication: remove intervals overlapping within 3 seconds
        is_duplicate = any(abs(start - s) <= 3 or abs(end - e) <= 3 for s, e in seen_intervals)
        
        if not is_duplicate:
            seen_intervals.append((start, end))
            bucket, key = task_info.get('s3_bucket', ''), task_info.get('s3_key', '')
            clips.append({
                'video': key, 
                's3_bucket': bucket, 
                'type': meta.get('embeddingOption'), 
                'timestamp': f'{start//60}:{start%60:02d}-{end//60}:{end%60:02d}', 
                'start_sec': start, 
                'end_sec': end,
                'similarity_score': v.get('distance', 0)
            })
    
    return {'query': query, 'clips': clips}

@tool
def get_clip_playback_url(s3_bucket: str, s3_key: str, start_sec: int, end_sec: int) -> Dict[str, Any]:
    '''Generate a presigned URL for playing a specific segment of an S3 video.'''
    base_url = s3_sigv4.generate_presigned_url('get_object', Params={'Bucket': s3_bucket, 'Key': s3_key}, ExpiresIn=3600)
    return {'playback_url': f'{base_url}#t={start_sec},{end_sec}'}

print('Search tools defined')

In [ ]:
# Summary tool
@tool
def summarize_video(s3_uri: str, prompt: str = 'Summarize this video in about 3 sentences, organized by chapters') -> Dict[str, Any]:
    '''Generate a video summary using the Pegasus model.'''
    response = bedrock_runtime.invoke_model(
        modelId=PEGASUS_MODEL_ID,
        body=json.dumps({
            'inputPrompt': prompt,
            'mediaSource': {'s3Location': {'uri': s3_uri, 'bucketOwner': ACCOUNT_ID}}
        }),
        contentType='application/json'
    )
    result = json.loads(response['body'].read())
    return {'s3_uri': s3_uri, 'summary': result.get('message', result.get('response', result))}

print('Summary tool defined')

In [ ]:
# Transcript tools
def _get_transcript_data(s3_uri: str):
    bucket = s3_uri.split('/')[2]
    job_name = f"transcript-{hashlib.md5(s3_uri.encode()).hexdigest()[:8]}"
    return json.loads(s3.get_object(Bucket=bucket, Key=f'transcripts/{job_name}.json')['Body'].read())

def _ensure_transcript(s3_uri: str) -> Dict[str, Any]:
    '''Check/create transcript file (internal).'''
    bucket = s3_uri.split('/')[2]
    job_name = f"transcript-{hashlib.md5(s3_uri.encode()).hexdigest()[:8]}"
    output_key = f'transcripts/{job_name}.json'
    try:
        status = transcribe.get_transcription_job(TranscriptionJobName=job_name)['TranscriptionJob']['TranscriptionJobStatus']
    except transcribe.exceptions.BadRequestException:
        key = '/'.join(s3_uri.split('/')[3:])
        ext = key.split('.')[-1].lower()
        media_format = 'mp4' if ext in ['mov', 'mp4', 'm4a'] else ext
        transcribe.start_transcription_job(TranscriptionJobName=job_name, Media={'MediaFileUri': s3_uri}, MediaFormat=media_format, LanguageCode='ko-KR', OutputBucketName=bucket, OutputKey=output_key)
        status = 'IN_PROGRESS'
    if status == 'IN_PROGRESS':
        print(f'Generating transcript... (job: {job_name})')
        while status == 'IN_PROGRESS':
            time.sleep(5)
            status = transcribe.get_transcription_job(TranscriptionJobName=job_name)['TranscriptionJob']['TranscriptionJobStatus']
    return {'transcript_file': f's3://{bucket}/{output_key}'} if status == 'COMPLETED' else {'error': 'Transcript generation failed'}

@tool
def get_transcript_file(s3_uri: str) -> Dict[str, Any]:
    '''Return the S3 path of the transcript file (creates one if it does not exist).'''
    return _ensure_transcript(s3_uri)

@tool
def get_transcript(s3_uri: str) -> Dict[str, Any]:
    '''Return the transcript in a timestamped table format. Creates one automatically if it does not exist.'''
    try:
        data = _get_transcript_data(s3_uri)
    except:
        result = _ensure_transcript(s3_uri)
        if 'error' in result:
            return result
        data = _get_transcript_data(s3_uri)
    grouped = {}
    for item in data['results']['items']:
        if item['type'] == 'pronunciation':
            sec = int(float(item.get('start_time', 0)))
            key = f"{sec//5*5//60}:{sec//5*5%60:02d}"
            grouped[key] = grouped.get(key, '') + ' ' + item['alternatives'][0]['content']
    return {'transcript': [{'time': k, 'text': v.strip()} for k, v in grouped.items()]}

@tool
def get_keywords(s3_uri: str) -> Dict[str, Any]:
    '''Extract keywords from the video transcript. Creates one automatically if it does not exist.'''
    try:
        data = _get_transcript_data(s3_uri)
    except:
        result = _ensure_transcript(s3_uri)
        if 'error' in result:
            return result
        data = _get_transcript_data(s3_uri)
    transcript = ' '.join([i['alternatives'][0]['content'] for i in data['results']['items'] if i['type'] == 'pronunciation'])[:2000]
    response = bedrock_runtime.invoke_model(modelId='us.anthropic.claude-3-5-haiku-20241022-v1:0',
        body=json.dumps({'anthropic_version': 'bedrock-2023-05-31', 'max_tokens': 256,
            'messages': [{'role': 'user', 'content': f'Extract 10 key keywords as a JSON array only:\n{transcript}'}]}),
        contentType='application/json')
    return {'keywords': json.loads(response['body'].read())['content'][0]['text']}

print('Transcript tools defined')

## Step 2: Create Specialized Agents

In [ ]:
# Video Analysis Agent (embedding + summarization)
video_analysis_agent = Agent(
    model=claude_model,
    tools=[create_video_embedding, summarize_video],
    system_prompt='You are a video analysis specialist agent. You handle video embedding creation and summarization.'
)

# Search Agent (search + playback URL)
search_agent = Agent(
    model=claude_model,
    tools=[search_video_clips, get_clip_playback_url],
    system_prompt='You are a video search specialist agent. You handle video clip search and playback URL generation. When search results are found, automatically generate playback URLs as well.'
)

# Transcript Agent (transcript + keywords)
transcript_agent = Agent(
    model=claude_model,
    tools=[get_transcript_file, get_transcript, get_keywords],
    system_prompt='You are a transcript processing specialist agent. You handle transcript creation, retrieval, and keyword extraction.'
)

print('Specialized agents created')

## Step 3: Wrap Agents as Tools

In [ ]:
@tool
def video_analysis_tool(query: str) -> str:
    """
    Video analysis tool (embedding creation, summarization)
    
    Use this tool for video embedding creation and summarization.
    
    Args:
        query: The analysis request (String)
    
    Returns:
        Analysis results (String)
    """
    print("Video Analysis Agent invoked")
    response = video_analysis_agent(query)
    print(f"Video Analysis Agent response complete")
    return str(response)

@tool
def video_search_tool(query: str) -> str:
    """
    Video search tool (clip search, playback URL generation)
    
    Use this tool for searching video clips and generating playback URLs.
    
    Args:
        query: The search request (String)
    
    Returns:
        Search results with playback links (String)
    """
    print("Video Search Agent invoked")
    response = search_agent(query)
    print(f"Video Search Agent response complete")
    return str(response)

@tool
def transcript_tool(query: str) -> str:
    """
    Transcript processing tool (transcript creation/retrieval, keyword extraction)
    
    Use this tool for transcript generation, retrieval, and keyword extraction.
    
    Args:
        query: The transcript request (String)
    
    Returns:
        Transcript or keyword results (String)
    """
    print("Transcript Agent invoked")
    response = transcript_agent(query)
    print(f"Transcript Agent response complete")
    return str(response)

print('Agent as Tool wrapping complete')

## Step 4: Create the Orchestrator

In [ ]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are the Video Processing Orchestrator Agent.

Your key responsibilities:
1. Use video_analysis_tool for video embedding creation and summarization
2. Use video_search_tool for searching video clips and generating playback URLs
3. Use transcript_tool for transcript generation, retrieval, and keyword extraction
4. Coordinate between different agents to provide comprehensive video analysis

Tool Selection Guidelines:
- For embedding creation or video summarization -> video_analysis_tool
- For searching clips or goal scenes -> video_search_tool
- For transcripts, subtitles, or keywords -> transcript_tool
- For complex requests, use multiple tools in sequence

Always provide clear, helpful responses to users.
"""

orchestrator = Agent(
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    tools=[video_analysis_tool, video_search_tool, transcript_tool],
    model=claude_model,
    callback_handler=PrintingCallbackHandler(),
    conversation_manager=SlidingWindowConversationManager(window_size=20, should_truncate_results=True)
)

print('Agent as Tool pattern fully implemented')

## Step 5: Test Run

In [ ]:
# Test 1: Search for goal scenes (up to 10)
#response = orchestrator("Find 10 goal-scoring scenes")
#print(response)

In [ ]:
# Test 2: Complex request (summarize + keywords + search)
response = orchestrator(f"Embed {VIDEO_S3_URI} first. Then retrieve exciting ceremonial scenes.")
print(response)